In [1]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_anthropic.chat_models import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
#from google.colab import userdata
from langchain_core.tools import tool
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
import gradio as gr
import os
from PIL import Image
from collections import Counter
from typing import Annotated, TypedDict
import time, sys
from anthropic import Anthropic
from openai import OpenAI

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sys.path.append('code')
from MolPropOp import *
#from lipinski_module import *
from docking_module import *

## adversary test

In [ ]:
tools = [grow_cycle, replace_groups, make_random_list, related, lipinski]

anthropic_key = os.getenv("ANTHROPIC_KEY")
client = Anthropic(api_key=anthropic_key)

#openai_key = os.getenv("OPENAI_API_TOKEN")
#model = ChatOpenAI(model_name="gpt-5.2", api_key=openai_key).bind_tools(tools)

google_key = os.getenv('GEMINI_API')
model =  ChatGoogleGenerativeAI(
    model='gemini-3-flash-preview', api_key=google_key).bind_tools(tools)

def adversary(prompt: str):
    adversary_message = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=2000,
    system = '''
    You are a drug design assistant. You will recieve a proposal from  another model
    of novel molecules it has designed to bind to a particular protein target. The proposal will 
    include reasoning as to why the model thinks those molecules will bind well, and estimated 
    docking scores for each molecule. Your task is to analyze the proposal and find any flaws 
    in the reasoning or estimation of the docking scores. You should then suggest modifications 
    to the proposed molecules that would make them more likely to bind well, and provide reasoning 
    for why those modifications would help.

    The other model has access to the following tools, and you may suggest that it use these tools to 
    gather more information or test out modifications to the proposed molecules:

    - grow_cycle: starts with a molecule SMILES and adds substituents to it, docks them, and returns 
                  a list of molecules and scores. 

    - replace_groups: starts with a molecule SMILES and replaces specific groups in it with new groups, 
                      returning a list of new molecules and scores. 
    
    - make_random_list: this tool generates a list of substituents of specified length (num_items). 
    
    - related: this tool generates a list of molecules that are structurally related to a given molecule, 
                and may be useful for exploring the chemical space around promising molecules.

    - lipinski: this tool evaluates a list of molecules for their drug-likeness based on Lipinski's rule of five.
    ''',
    messages=[
        {"role": "user", "content": prompt},])

    response = adversary_message.content[0].text

    return response

class State(TypedDict):
  messages: Annotated[list, add_messages]

def model_node(state: State) -> State:
  res = model.invoke(state['messages'])
  return {'messages': res}

builder = StateGraph(State)
builder.add_node('model', model_node)
builder.add_node('tools', ToolNode(tools))
builder.add_edge(START, 'model')
builder.add_conditional_edges('model', tools_condition)
builder.add_edge('tools',  'model')

graph = builder.compile()

sys_message = SystemMessage(content=f'''
{task_specific_prompt}

## You will first:
- Read the list of molecule SMILES and scores
- Ascertain any features of the molecules that contribute to the desired score. For example, if,
from one molecule to the next, the addition of an O group makes the score better.
- Gather all of these trends across all of the molecules.

## If you need additional information to ascertain the trends, such as more modified
molecules and their docking scores, you have tools you can call to generate new
molecules and get their docking scores. You can use these tools as many times as you want
to gather information on the trends. *NOTE: if you choose to add a phenyl group to a molecule,
use the SMILES 'c7ccccc7', so that it does not interfere with other rings in the molecule that
may already use numbers 1-6 in their SMILES notation. 

The tools you have available include:
                            
- grow_cycle: starts with a molecule SMILES and adds substituents to it, docks them, and returns 
              a list of molecules and scores. You can use this tool to further explore modifications
              to promising molecules that you find in the input data. You can provide a list of
              substituents to add, or use the predefined sets: e_withdraw (electron withdrawing),
              e_donate (electron donating), withdraw_with_linkers (electron withdrawing with linkers), 
              donate_with_linkers (electron donating with linkers). You can also generate a random list 
              of substituents with the make_random_list tool and use that as input to grow_cycle.

- replace_groups: starts with a molecule SMILES and replaces specific groups in it with new groups, returning a list of new
                  molecules and scores. This tool allows you to test specific hypotheses about how replacing certain
                  groups in a molecule might affect binding affinity. You can specify which groups to replace and
                  what to replace them with, or use the predefined sets of substituents mentioned above. You can also 
                  generate a random list of substituents with the make_random_list tool and use that as input to replace_groups.

- make_random_list: this tool generates a list of substituents of specified length (num_items). It draws from the predefined lists:
                    e_withdraw (electron withdrawing), e_donate (electron donating), withdraw_with_linkers 
                    (electron withdrawing with linkers), donate_with_linkers (electron donating with linkers). 
                    Use this tool when you want to get a broad sense of how different modifications affect binding affinity, 
                    without having a specific hypothesis in mind.

- related: this tool generates a list of molecules that are structurally related to a given molecule, and
           may be useful for exploring the chemical space around promising molecules you find in the input 
           data. It returns a list of related molecules and a few properties.

- lipinski: this tool evaluates a list of molecules for their drug-likeness based on Lipinski's rule of five, 
            which is a set of guidelines for determining whether a molecule is likely to be an orally active 
            drug in humans. This tool can help you ensure that the molecules you are proposing not only have 
            good docking scores but also have properties that make them more likely to be successful as drugs.
            QED (quantitative estimate of drug-likeness) is a score between 0 and 1 that summarizes how 
            drug-like a molecule is, with 1 being the most drug-like. A higher QED score indicates that a 
            molecule has properties that are more consistent with known drugs, such as appropriate molecular 
            weight, lipophilicity, and number of hydrogen bond donors and acceptors.

## Once you have ascertained the trends:
- Use the trends you learned to suggest 1-5 new molecules that obey the trends you found
and which should have a better score than the molecules in the list.
- Provide reasoning as to why you created those new molecules.
- Estimate the new scores.

## You may ask the user for clarification if needed, but try to use the tools to gather as much information as you
can before asking for clarification.

## In further turns, you will also receive feedback from an adversary model that is trying to find flaws 
in your reasoning and suggest improvements to your proposed molecules. You should use this feedback to 
refine your understanding of the trends, run new experiments with the tools to gather more information, 
and improve your proposed molecules in subsequent turns.

## If you have identified good potential hits, evaluate the Lipinski properties of the proposed molecules 
and use that information to further refine your proposals, keeping in mind that you want to propose molecules 
that not only have good docking scores but also have good drug-like properties. 

## Once you have reached a point where you think you have proposed the best possible molecules based on 
the trends, tool results and the adversary feedback, reply with only one word: "Done". This will signal that you have 
finished the task and will not propose any more molecules.
''')

global messages
messages = [sys_message]

#@spaces.GPU
def start_chat():
  '''
  '''
  global chat_history, messages, reasoning
  chat_history = []
  reasoning = []
  messages = [sys_message]

#@spaces.GPU
def chat_turn(prompt: str):
  '''
  '''
  global chat_history, messages, reasoning
  human_message = HumanMessage(content=prompt)
  messages.append(human_message)
  local_history = [prompt]

  input = {
      'messages' : messages
  }

  for c in graph.stream(input):
    try:
      ai_mes = c['model']['messages'].content
      messages.append(AIMessage(ai_mes))
      if ai_mes != '':
        #print(f'message is {ai_mes}')
        local_history.append(ai_mes)
    except:
      pass
    try:
      if os.path.exists('current_image.png'):
        if os.path.getmtime('current_image.png') > time.time() - 30:
          img = Image.open('current_image.png')
        else:
          img = None
      else:
        img = None
    except:
      img = None
    try:
      reasoning.append(c['tools']['messages'][0].content)
    except:
      pass

  if len(local_history) != 2:
    local_history.append('no message')

  #chat_history.append({'role': 'user', 'content': local_history[0]})
  #chat_history.append({'role': 'assistant', 'content': local_history[1]})
  chat_history.append(local_history)
  return '', img, chat_history

def get_initial_prompt(protein):
  '''
  '''
  scoring_args[1] = protein
  #random_list, excluded_list = make_random_list(10)
  #first_list = sub_cycle(random_list, scoring_args)
  #context = ''
  #for smiles, score in first_list:
  #    context += f"{smiles}: {score}\n"

  with open('adversarial_set.md', 'r') as f:
    context = f.read()
  first_prompt = f'''
  Here is a list of molecules and their docking scores:
  {context}\n'''

  blank, img, mes = chat_turn(first_prompt)
  #print(mes[-1])
  
  #chat_history.append({'role': 'user', 'content': 'User sent protein name'})
  #chat_history.append({'role': 'assistant', 'content': 'Assistant running sub_cycle'})
  #chat_history.append({'role': 'user', 'content': first_prompt})
  #chat_history.append({'role': 'assistant', 'content': mes[-1]})

  return mes



In [6]:
messages

[SystemMessage(content='\n# You are a drug design assistant. In the first user message you will\nsee a list of molecule SMILES strings and docking scores.\nThe lower the docking score (the more negative), the more affinity the\nmolecule has for the protein in question. Your task is to use the information \nin the list to learn trends about what makes a molecule a good binder, and then \nuse those trends to suggest new molecules that should have better docking scores \n(more negative) than the ones in the list.\n\n## You will first:\n- Read the list of molecule SMILES and scores\n- Ascertain any features of the molecules that contribute to the desired score. For example, if,\nfrom one molecule to the next, the addition of an O group makes the score better.\n- Gather all of these trends across all of the molecules.\n\n## If you need additional information to ascertain the trends, such as more modified\nmolecules and their docking scores, you have tools you can call to generate new\nmolec

In [ ]:
start_chat()

date_string = time.strftime("%Y-%m-%d_%H-%M-%S")
filename = f'../results/GEMINI_FIRST/adversary_design_{date_string}.md'
with open(filename, 'w') as f:
    f.write(f'# Adversarial Design Session - {date_string}\n\n')

response_list = get_initial_prompt('HMGCR')
with open(filename, 'a') as f:
    f.write('# Initial model response:\n')
    text_av = response_list[-1][-2][-1]['text']+'\n'
    f.write(text_av)

text_av = ''
while text_av != 'Done':

    ant_response = adversary(response_list[-1][-2][-1]['text'])
    with open(filename, 'a') as f:
        f.write('\n# Adversary feedback:\n')
        text_av = ant_response+'\n'
        f.write(text_av)

    _, _, response_list = chat_turn(ant_response)
    with open(filename, 'a') as f:
        f.write('\n# Model response:\n')
        text_av = response_list[-1][-2][-1]['text']+'\n'
        f.write(text_av)

lipinski tool
related tool
Starting grow cycle with best score -8.6 for O=c1cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12.
O=c1(F)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(Cl)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(O)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(N)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(F)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(Cl)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(O)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(N)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2(F)cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2(Cl)cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2(O)cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2(N)cccc(C(C(=O)[O-]))c12 bad


[08:21:28] Explicit valence for atom # 1 C, 6, is greater than permitted
[08:21:28] Explicit valence for atom # 1 C, 6, is greater than permitted
[08:21:28] Explicit valence for atom # 1 C, 6, is greater than permitted
[08:21:28] Explicit valence for atom # 1 C, 6, is greater than permitted
[08:21:28] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[08:21:28] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[08:21:28] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[08:21:28] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[08:21:28] Can't kekulize mol.  Unkekulized atoms: 2 3 13 14 15 16 21
[08:21:28] Can't kekulize mol.  Unkekulized atoms: 2 3 13 14 15 16 21
[08:21:28] Can't kekulize mol.  Unkekulized atoms: 2 3 13 14 15 16 21
[08:21:28] Can't kekulize mol.  Unkekulized atoms: 2 3 13 14 15 16 21


got related molecules with smiles
(8-acetyloxy-9-oxo-10H-anthracen-1-yl) acetate
O=c1c(F)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -8.3
O=c1c(Cl)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.4
O=c1c(O)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -8.4
O=c1c(N)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.9
O=c1cc(-c2c(F)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2c(Cl)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2c(O)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2c(N)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.4
O=c1cc(-c2cc(F)ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2cc(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2cc(O)ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.4
O=c1cc(-c2cc(N)ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.4
O=c1cc(-c2ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.7
=========== New best score: -8.7 for O=c1cc(-c2ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12 ========
O=c1cc(-c2ccc(Cl)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2ccc(O)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.0
O=c1cc(-c2ccc(N)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-

[08:28:47] Explicit valence for atom # 17 O, 2, is greater than permitted
[08:28:47] Explicit valence for atom # 17 P, 6, is greater than permitted


Starting grow cycle with best score -8.7 for O=c1cc(-c2ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12.
O=c1(F)cc(-c2ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(O)cc(-c2ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(N)cc(-c2ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(F)ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(O)ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(N)ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccc(F)cc2)oc2(F)cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccc(F)cc2)oc2(O)cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccc(F)cc2)oc2(N)cccc(C(C(=O)[O-]))c12 bad


...

O=c1c(F)c(-c2ccccc2)oc2cccc(CC(=O)[O-])c12: -8.3
O=c1c(Cl)c(-c2ccccc2)oc2cccc(CC(=O)[O-])c12: -7.4
O=c1c(OC)c(-c2ccccc2)oc2cccc(CC(=O)[O-])c12: -7.5
O=c1c(N)c(-c2ccccc2)oc2cccc(CC(=O)[O-])c12: -7.9
O=c1cc(-c2c(F)cccc2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2c(Cl)cccc2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2c(OC)cccc2)oc2cccc(CC(=O)[O-])c12: -8.4
O=c1cc(-c2c(N)cccc2)oc2cccc(CC(=O)[O-])c12: -8.4
O=c1cc(-c2cc(F)ccc2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2cc(Cl)ccc2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2cc(OC)ccc2)oc2cccc(CC(=O)[O-])c12: -8.4
O=c1cc(-c2cc(N)ccc2)oc2cccc(CC(=O)[O-])c12: -8.4
O=c1cc(-c2ccc(F)cc2)oc2cccc(CC(=O)[O-])c12: -8.7
O=c1cc(-c2ccc(Cl)cc2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2ccc(OC)cc2)oc2cccc(CC(=O)[O-])c12: -7.8
O=c1cc(-c2ccc(N)cc2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2cccc(F)c2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2cccc(Cl)c2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2cccc(OC)c2)oc2cccc(CC(=O)[O-])c12: -8.4
O=c1cc(-c2cccc(N)c2)oc2cccc(CC(=O)[O-])c12: -8.4
O=c1cc(-c2

[10:52:09] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:52:09] Can't kekulize mol.  Unkekulized atoms: 6 7 9 10 11
[10:52:09] Can't kekulize mol.  Unkekulized atoms: 2 3 14 15 16 17 22


O=c1c(F)c(-c2cc(F)ccc2)oc2cccc(CC(=O)[O-])c12: -8.3
O=c1cc(-c2c(F)c(F)ccc2)oc2cccc(CC(=O)[O-])c12: -8.8
O=c1cc(-c2cc(F)c(F)cc2)oc2cccc(CC(=O)[O-])c12: -8.9
O=c1cc(-c2cc(F)cc(F)c2)oc2cccc(CC(=O)[O-])c12: -8.0
O=c1cc(-c2cc(F)ccc2)oc2c(F)ccc(CC(=O)[O-])c12: -8.0
O=c1cc(-c2cc(F)ccc2)oc2cc(F)cc(CC(=O)[O-])c12: -8.8
O=c1cc(-c2cc(F)ccc2)oc2ccc(F)c(CC(=O)[O-])c12: -8.0
Best score: -8.9 for O=c1cc(-c2cc(F)ccc2)oc2cccc(CC(=O)[O-])c12
Starting grow cycle with best score -8.6 for O=c1cc(-c2ccccc2)oc2cccc(CC(=O)[O-])c12.
O=c1(F)cc(-c2ccccc2)oc2cccc(CC(=O)[O-])c12 bad
O=c1(Cl)cc(-c2ccccc2)oc2cccc(CC(=O)[O-])c12 bad
O=c1(OC)cc(-c2ccccc2)oc2cccc(CC(=O)[O-])c12 bad
O=c1(N)cc(-c2ccccc2)oc2cccc(CC(=O)[O-])c12 bad
O=c1([N+](=O)[O-])cc(-c2ccccc2)oc2cccc(CC(=O)[O-])c12 bad
O=c1cc(-c2(F)ccccc2)oc2cccc(CC(=O)[O-])c12 bad
O=c1cc(-c2(Cl)ccccc2)oc2cccc(CC(=O)[O-])c12 bad
O=c1cc(-c2(OC)ccccc2)oc2cccc(CC(=O)[O-])c12 bad
O=c1cc(-c2(N)ccccc2)oc2cccc(CC(=O)[O-])c12 bad
O=c1cc(-c2([N+](=O)[O-])ccccc2)oc2cccc(CC(=O)[O-

[10:54:15] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:54:15] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:54:15] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:54:15] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:54:15] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:54:15] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[10:54:15] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[10:54:15] Can't kekulize mol.  Unkekulized atoms: 7 8 9 10 11
[10:54:15] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[10:54:15] Can't kekulize mol.  Unkekulized atoms: 8 9 10 11 12
[10:54:15] Can't kekulize mol.  Unkekulized atoms: 2 3 13 14 15 16 21
[10:54:15] Can't kekulize mol.  Unkekulized atoms: 2 3 13 14 15 16 21
[10:54:15] Can't kekulize mol.  Unkekulized atoms: 2 3 14 15 16 17 22
[10:54:15] Can't kekulize mol.  Unkekulized atoms: 2 3 13 14 15 16 21
[10:54:15] Can't kekulize mol.  Unkekulize

O=c1c(F)c(-c2ccccc2)oc2cccc(CC(=O)[O-])c12: -8.3
O=c1c(Cl)c(-c2ccccc2)oc2cccc(CC(=O)[O-])c12: -7.4
O=c1c(OC)c(-c2ccccc2)oc2cccc(CC(=O)[O-])c12: -7.5
O=c1c(N)c(-c2ccccc2)oc2cccc(CC(=O)[O-])c12: -7.9
O=c1c([N+](=O)[O-])c(-c2ccccc2)oc2cccc(CC(=O)[O-])c12: -7.0
O=c1cc(-c2c(F)cccc2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2c(Cl)cccc2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2c(OC)cccc2)oc2cccc(CC(=O)[O-])c12: -8.4
O=c1cc(-c2c(N)cccc2)oc2cccc(CC(=O)[O-])c12: -8.4
O=c1cc(-c2c([N+](=O)[O-])cccc2)oc2cccc(CC(=O)[O-])c12: -8.3
O=c1cc(-c2cc(F)ccc2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2cc(Cl)ccc2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2cc(OC)ccc2)oc2cccc(CC(=O)[O-])c12: -8.4
O=c1cc(-c2cc(N)ccc2)oc2cccc(CC(=O)[O-])c12: -8.4
O=c1cc(-c2cc([N+](=O)[O-])ccc2)oc2cccc(CC(=O)[O-])c12: -8.2
O=c1cc(-c2ccc(F)cc2)oc2cccc(CC(=O)[O-])c12: -8.7
=========== New best score: -8.7 for O=c1cc(-c2ccc(F)cc2)oc2cccc(CC(=O)[O-])c12 ========
O=c1cc(-c2ccc(Cl)cc2)oc2cccc(CC(=O)[O-])c12: -8.6
O=c1cc(-c2ccc(OC)cc2)oc2cccc(CC(=O)[O-

## FIX ANT

In [8]:
tools = [grow_cycle, replace_groups, make_random_list, related, lipinski]

anthropic_key = os.getenv("ANTHROPIC_KEY")
model = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=anthropic_key).bind_tools(tools)

openai_key = os.getenv("OPENAI_API_TOKEN")
client = OpenAI(api_key=openai_key)

def adversary(prompt: str):
    
    adversary_message = client.responses.create(
      model="gpt-5.2",
      instructions = '''
      You are a drug design assistant. You will recieve a proposal from  another model
      of novel molecules it has designed to bind to a particular protein target. The proposal will 
      include reasoning as to why the model thinks those molecules will bind well, and estimated 
      docking scores for each molecule. Your task is to analyze the proposal and find any flaws 
      in the reasoning or estimation of the docking scores. You should then suggest modifications 
      to the proposed molecules that would make them more likely to bind well, and provide reasoning 
      for why those modifications would help.

      The other model has access to the following tools, and you may suggest that it use these tools to 
      gather more information or test out modifications to the proposed molecules:

      - grow_cycle: starts with a molecule SMILES and adds substituents to it, docks them, and returns 
                    a list of molecules and scores. 

      - replace_groups: starts with a molecule SMILES and replaces specific groups in it with new groups, 
                        returning a list of new molecules and scores. 
      
      - make_random_list: this tool generates a list of substituents of specified length (num_items). 
      
      - related: this tool generates a list of molecules that are structurally related to a given molecule, 
                  and may be useful for exploring the chemical space around promising molecules.

      - lipinski: this tool evaluates a list of molecules for their drug-likeness based on Lipinski's rule of five.
      ''',
      input=prompt
    )
    
    response = adversary_message.output_text

    return response

class State(TypedDict):
  messages: Annotated[list, add_messages]

def model_node(state: State) -> State:
  res = model.invoke(state['messages'])
  return {'messages': res}

builder = StateGraph(State)
builder.add_node('model', model_node)
builder.add_node('tools', ToolNode(tools))
builder.add_edge(START, 'model')
builder.add_conditional_edges('model', tools_condition)
builder.add_edge('tools',  'model')

graph = builder.compile()

sys_message = SystemMessage(content=f'''
{task_specific_prompt}

## You will first:
- Read the list of molecule SMILES and scores
- Ascertain any features of the molecules that contribute to the desired score. For example, if,
from one molecule to the next, the addition of an O group makes the score better.
- Gather all of these trends across all of the molecules.

## If you need additional information to ascertain the trends, such as more modified
molecules and their docking scores, you have tools you can call to generate new
molecules and get their docking scores. You can use these tools as many times as you want
to gather information on the trends. *NOTE: if you choose to add a phenyl group to a molecule,
use the SMILES 'c7ccccc7', so that it does not interfere with other rings in the molecule that
may already use numbers 1-6 in their SMILES notation. 

The tools you have available include:
                            
- grow_cycle: starts with a molecule SMILES and adds substituents to it, docks them, and returns 
              a list of molecules and scores. You can use this tool to further explore modifications
              to promising molecules that you find in the input data. You can provide a list of
              substituents to add, or use the predefined sets: e_withdraw (electron withdrawing),
              e_donate (electron donating), withdraw_with_linkers (electron withdrawing with linkers), 
              donate_with_linkers (electron donating with linkers). You can also generate a random list 
              of substituents with the make_random_list tool and use that as input to grow_cycle.

- replace_groups: starts with a molecule SMILES and replaces specific groups in it with new groups, returning a list of new
                  molecules and scores. This tool allows you to test specific hypotheses about how replacing certain
                  groups in a molecule might affect binding affinity. You can specify which groups to replace and
                  what to replace them with, or use the predefined sets of substituents mentioned above. You can also 
                  generate a random list of substituents with the make_random_list tool and use that as input to replace_groups.

- make_random_list: this tool generates a list of substituents of specified length (num_items). It draws from the predefined lists:
                    e_withdraw (electron withdrawing), e_donate (electron donating), withdraw_with_linkers 
                    (electron withdrawing with linkers), donate_with_linkers (electron donating with linkers). 
                    Use this tool when you want to get a broad sense of how different modifications affect binding affinity, 
                    without having a specific hypothesis in mind.

- related: this tool generates a list of molecules that are structurally related to a given molecule, and
           may be useful for exploring the chemical space around promising molecules you find in the input 
           data. It returns a list of related molecules and a few properties.

- lipinski: this tool evaluates a list of molecules for their drug-likeness based on Lipinski's rule of five, 
            which is a set of guidelines for determining whether a molecule is likely to be an orally active 
            drug in humans. This tool can help you ensure that the molecules you are proposing not only have 
            good docking scores but also have properties that make them more likely to be successful as drugs.
            QED (quantitative estimate of drug-likeness) is a score between 0 and 1 that summarizes how 
            drug-like a molecule is, with 1 being the most drug-like. A higher QED score indicates that a 
            molecule has properties that are more consistent with known drugs, such as appropriate molecular 
            weight, lipophilicity, and number of hydrogen bond donors and acceptors.

## Once you have ascertained the trends:
- Use the trends you learned to suggest 1-5 new molecules that obey the trends you found
and which should have a better score than the molecules in the list.
- Provide reasoning as to why you created those new molecules.
- Estimate the new scores.

## You may ask the user for clarification if needed, but try to use the tools to gather as much information as you
can before asking for clarification.

## In further turns, you will also receive feedback from an adversary model that is trying to find flaws 
in your reasoning and suggest improvements to your proposed molecules. You should use this feedback to 
refine your understanding of the trends, run new experiments with the tools to gather more information, 
and improve your proposed molecules in subsequent turns.

## If you have identified good potential hits, evaluate the Lipinski properties of the proposed molecules 
and use that information to further refine your proposals, keeping in mind that you want to propose molecules 
that not only have good docking scores but also have good drug-like properties. 

## Once you have reached a point where you think you have proposed the best possible molecules based on 
the trends, tool results and the adversary feedback, reply with only one word: "Done". This will signal that you have 
finished the task and will not propose any more molecules.
''')

global messages
messages = [sys_message]

#@spaces.GPU
def start_chat():
  '''
  '''
  global chat_history, messages, reasoning
  chat_history = []
  reasoning = []
  messages = [sys_message]

#@spaces.GPU
def chat_turn(prompt: str):
  '''
  '''
  global chat_history, messages, reasoning
  human_message = HumanMessage(content=prompt)
  messages.append(human_message)
  local_history = [prompt]

  input = {
      'messages' : messages
  }

  for c in graph.stream(input):
    try:
      ai_mes = c['model']['messages'].content
      #print(c)
      messages.append(AIMessage(ai_mes))
      if ai_mes != '':
        #print(f'message is {ai_mes}')
        local_history.append(ai_mes)
    except:
      pass
    try:
      if os.path.exists('current_image.png'):
        if os.path.getmtime('current_image.png') > time.time() - 30:
          img = Image.open('current_image.png')
        else:
          img = None
      else:
        img = None
    except:
      img = None
    try:
      reasoning.append(c['tools']['messages'][0].content)
    except:
      pass

  if len(local_history) != 2:
    local_history.append('no message')

  #chat_history.append({'role': 'user', 'content': local_history[0]})
  #chat_history.append({'role': 'assistant', 'content': local_history[1]})
  chat_history.append(local_history)
  return '', img, chat_history

def get_initial_prompt(protein):
  '''
  '''
  scoring_args[1] = protein
  #random_list, excluded_list = make_random_list(10)
  #first_list = sub_cycle(random_list, scoring_args)
  #context = ''
  #for smiles, score in first_list:
  #    context += f"{smiles}: {score}\n"

  with open('adversarial_set.md', 'r') as f:
    context = f.read()
  first_prompt = f'''
  Here is a list of molecules and their docking scores:
  {context}\n'''

  blank, img, mes = chat_turn(first_prompt)
  #print(mes[-1])
  
  #chat_history.append({'role': 'user', 'content': 'User sent protein name'})
  #chat_history.append({'role': 'assistant', 'content': 'Assistant running sub_cycle'})
  #chat_history.append({'role': 'user', 'content': first_prompt})
  #chat_history.append({'role': 'assistant', 'content': mes[-1]})

  return mes


In [36]:
response_list

[['\n  Here is a list of molecules and their docking scores:\n  Starting sub cycle on base rings for protein HMGCR.\nc1(N(I))ccccc1: -5.1\nc1(O(C#N))ccccc1: -4.9\nc1(C(=O)O(C(C)C))ccccc1: -5.3\nc1(C#C(SC))ccccc1: -5.2\nc1(C(C(=O)[O-]))ccccc1: -5.4\nc1(C(C))ccccc1: -4.9\nc1(C=C([N+](=O)[O-]))ccccc1: -5.8\nc1(C(N))ccccc1: -5.0\nc1(C([O-]))ccccc1: -4.9\nc1(CC(N(C)C))ccccc1: -5.5\nn1c(N(I))cccc1: -5.1\nn1c(O(C#N))cccc1: -5.1\nn1c(C(=O)O(C(C)C))cccc1: -6.1\nn1c(C#C(SC))cccc1: -4.9\nn1c(C(C(=O)[O-]))cccc1: -5.7\nn1c(C(C))cccc1: -4.3\nn1c(C=C([N+](=O)[O-]))cccc1: -5.3\nn1c(C(N))cccc1: -4.9\nn1c(C([O-]))cccc1: -4.9\nn1c(CC(N(C)C))cccc1: -5.5\nn1cc(N(I))ccc1: -5.1\nn1cc(O(C#N))ccc1: -4.9\nn1cc(C(=O)O(C(C)C))ccc1: -5.2\nn1cc(C#C(SC))ccc1: -5.2\nn1cc(C(C(=O)[O-]))ccc1: -5.6\nn1cc(C(C))ccc1: -5.0\nn1cc(C=C([N+](=O)[O-]))ccc1: -5.4\nn1cc(C(N))ccc1: -4.4\nn1cc(C([O-]))ccc1: -5.1\nn1cc(CC(N(C)C))ccc1: -5.5\nn1ccc(N(I))cc1: -4.4\nn1ccc(O(C#N))cc1: -4.7\nn1ccc(C(=O)O(C(C)C))cc1: -5.4\nn1ccc(C#C(SC))cc1

In [29]:
text_av = response_list[-1][-2]

In [79]:
text_av

'Excellent! Now I have genuine, tool-validated insights. Let me compile my revised, honest final proposals:\n\n---\n\n## REVISED FINAL PROPOSALS (Tool-Validated)\n\n### **Rank 1: O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12**\n- **Docking Score: -8.9 kcal/mol** (replicated across multiple analogs)\n- **QED: 0.662** (Good drug-likeness)\n- **MW: 342.3, LogP: 1.503** (Excellent for oral absorption)\n- **PSA: 136.6, HBD: 3, HBA: 5** (Acceptable)\n- **Rotatable bonds: 3** (Low flexibility)\n- **No undesirable moieties** ✓\n- **Design logic (validated by tool):**\n  - Chromone hydroxyl at position 2 is genuinely critical (+0.9 kcal/mol when masked)\n  - Dual primary amides outperform carboxylates by ~0.7 kcal/mol (H-bonding geometry > ionic anchor)\n  - Fluorine at chromone position 4 consistently improves binding (-8.9 with F vs -8.4 without)\n  - Phenyl amide at para position scores as well as ortho/meta variants\n- **Limitations:** Still high PSA and 3 HBD; likely requires facilitat

In [80]:
filename = '../results/ANT_FIRST/adversary_design_2026-04-01_10-27-25.md'

adv_response = adversary(text_av)
with open(filename, 'a') as f:
    f.write('\n# Adversary feedback:\n')
    text_av = adv_response
    f.write(text_av +'\n')

In [ ]:
#_ = messages.pop(8)
messages

[SystemMessage(content='\n# You are a drug design assistant. In the first user message you will\nsee a list of molecule SMILES strings and docking scores.\nThe lower the docking score (the more negative), the more affinity the\nmolecule has for the protein in question. Your task is to use the information \nin the list to learn trends about what makes a molecule a good binder, and then \nuse those trends to suggest new molecules that should have better docking scores \n(more negative) than the ones in the list.\n\n## You will first:\n- Read the list of molecule SMILES and scores\n- Ascertain any features of the molecules that contribute to the desired score. For example, if,\nfrom one molecule to the next, the addition of an O group makes the score better.\n- Gather all of these trends across all of the molecules.\n\n## If you need additional information to ascertain the trends, such as more modified\nmolecules and their docking scores, you have tools you can call to generate new\nmolec

In [93]:
adv_response

'### Main issues / likely flaws in the reasoning and the “-8.9 kcal/mol tie” conclusions\n\n1. **Score replication across close analogs is not strong evidence of true equivalence**\n   - Seeing **F vs Cl “indistinguishable”** at exactly the same score is a red flag for **score quantization / limited precision** in the docking setup (grid, sampling, scoring function resolution). In many workflows, differences of **≤0.5–1.0 kcal/mol** are within noise unless you have multiple independent runs *and* pose clustering consistency.\n   - Recommendation: treat Rank 1–3 as a **single chemotype cluster** with uncertain internal ordering.\n\n2. **The “+0.9 kcal/mol when masked” hydroxyl claim may be conflating pose changes with true interaction energy**\n   - Converting **phenolic OH → OMe** changes more than HBD ability: it changes **preferred tautomer/rotamer, desolvation penalty, and sterics**, often producing a different pose. A worse score could come from a **pose flip** rather than loss of 

In [60]:
print(scoring_args[1])
#score, _ = scoring_function('O=c1c(O)c(-c2c(C(=O)[O-])c(NC(=O)C)c(F)cc2)oc2cc(F)cc(C)c12')
#print(score)

HMGCR


In [94]:
# messages = [
#     sys_message,
#     HumanMessage(content=initial_data),
#     AIMessage(content=mr1),
# ]
filename = '../results/ANT_FIRST/adversary_design_2026-04-01_10-27-25.md'
_, _, response_list = chat_turn(adv_response)
with open(filename, 'a') as f:
    f.write('\n# Model response:\n')
    text_av = response_list[-1][-2]
    f.write(text_av +'\n')

Starting replace cycle with best score 0.0 for O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12.
Found O in O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12
OC=c1c(OC)c(-c2ccc(C(=OC)N)cc2)oc2c(F)ccc(C(=OC)N)c12 bad
OCC=c1c(OCC)c(-c2ccc(C(=OCC)N)cc2)oc2c(F)ccc(C(=OCC)N)c12 bad
OC(C)C=c1c(OC(C)C)c(-c2ccc(C(=OC(C)C)N)cc2)oc2c(F)ccc(C(=OC(C)C)N)c12 bad
Best score: 0.0 for O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12
lipinski tool

Starting replace cycle with best score 0.0 for O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12.
Found C(=O)N in O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12


[14:28:44] Explicit valence for atom # 12 O, 3, is greater than permitted
[14:28:44] Explicit valence for atom # 14 O, 3, is greater than permitted
[14:28:44] Explicit valence for atom # 16 O, 3, is greater than permitted


=========== New best score: -7.9 for O=c1c(O)c(-c2ccc(C(=O)NC)cc2)oc2c(F)ccc(C(=O)NC)c12 ========
=========== New best score: -8.3 for O=c1c(O)c(-c2ccc(C(=O)N(C)C)cc2)oc2c(F)ccc(C(=O)N(C)C)c12 ========


[14:29:43] SMILES Parse Error: syntax error while parsing: O=c1c(O)c(-c2ccc(C(=O)O-)cc2)oc2c(F)ccc(C(=O)O-)c12
[14:29:43] SMILES Parse Error: check for mistakes around position 25:
[14:29:43] c(O)c(-c2ccc(C(=O)O-)cc2)oc2c(F)ccc(C(=O)
[14:29:43] ~~~~~~~~~~~~~~~~~~~~^
[14:29:43] SMILES Parse Error: extra open parentheses while parsing: O=c1c(O)c(-c2ccc(C(=O)O-)cc2)oc2c(F)ccc(C(=O)O-)c12
[14:29:43] SMILES Parse Error: check for mistakes around position 10:
[14:29:43] O=c1c(O)c(-c2ccc(C(=O)O-)cc2)oc2c(F)ccc(C
[14:29:43] ~~~~~~~~~^
[14:29:43] SMILES Parse Error: extra open parentheses while parsing: O=c1c(O)c(-c2ccc(C(=O)O-)cc2)oc2c(F)ccc(C(=O)O-)c12
[14:29:43] SMILES Parse Error: check for mistakes around position 17:
[14:29:43] O=c1c(O)c(-c2ccc(C(=O)O-)cc2)oc2c(F)ccc(C
[14:29:43] ~~~~~~~~~~~~~~~~^
[14:29:43] SMILES Parse Error: Failed parsing SMILES 'O=c1c(O)c(-c2ccc(C(=O)O-)cc2)oc2c(F)ccc(C(=O)O-)c12' for input: 'O=c1c(O)c(-c2ccc(C(=O)O-)cc2)oc2c(F)ccc(C(=O)O-)c12'


O=c1c(O)c(-c2ccc(C(=O)O-)cc2)oc2c(F)ccc(C(=O)O-)c12 bad
Best score: -8.3 for O=c1c(O)c(-c2ccc(C(=O)N(C)C)cc2)oc2c(F)ccc(C(=O)N(C)C)c12
Starting replace cycle with best score 0.0 for O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12.
Found O in O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12
OC=c1c(OC)c(-c2ccc(C(=OC)N)cc2)oc2c(F)ccc(C(=OC)N)c12 bad
OCC=c1c(OCC)c(-c2ccc(C(=OCC)N)cc2)oc2c(F)ccc(C(=OCC)N)c12 bad
OCc1ccccc1=c1c(OCc1ccccc1)c(-c2ccc(C(=OCc1ccccc1)N)cc2)oc2c(F)ccc(C(=OCc1ccccc1)N)c12 bad
Best score: 0.0 for O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12
Starting grow cycle with best score -8.9 for O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12.
O=c1(C)c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12 bad
O=c1(CC)c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12 bad
O=c1(C(C)C)c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12 bad
O=c1(c1ccccc1)c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12 bad
O=c1(C(F)(F)F)c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12 bad
O=c1(OC)c(O)c(-c2ccc(C(=O)

...

O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)cc(C(F)(F)F)c(C(=O)N)c12: -7.9
O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)cc(OC)c(C(=O)N)c12: -7.9
O=c1c(O)c(-c2ccc(C(=O)N)cc2)oc2c(F)cc(N(C)C)c(C(=O)N)c12: -7.6
Best score: -9.2 for O=c1c(O)c(-c2c(C)cc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12
Starting replace cycle with best score 0.0 for O=c1c(O)c(-c2c(C)cc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12.
Found O in O=c1c(O)c(-c2c(C)cc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12
OC=c1c(OC)c(-c2c(C)cc(C(=OC)N)cc2)oc2c(F)ccc(C(=OC)N)c12 bad
OCC=c1c(OCC)c(-c2c(C)cc(C(=OCC)N)cc2)oc2c(F)ccc(C(=OCC)N)c12 bad
lipinski tool


[14:39:36] Explicit valence for atom # 13 O, 3, is greater than permitted
[14:39:36] Explicit valence for atom # 15 O, 3, is greater than permitted


=========== New best score: -6.5 for S=c1c(S)c(-c2c(C)cc(C(=S)N)cc2)oc2c(F)ccc(C(=S)N)c12 ========
Best score: -6.5 for S=c1c(S)c(-c2c(C)cc(C(=S)N)cc2)oc2c(F)ccc(C(=S)N)c12
Starting replace cycle with best score 0.0 for O=c1c(O)c(-c2c(C)cc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12.
Found F in O=c1c(O)c(-c2c(C)cc(C(=O)N)cc2)oc2c(F)ccc(C(=O)N)c12
O=c1c(O)c(-c2c(C)cc(C(=O)N)cc2)oc2c(H)ccc(C(=O)N)c12 bad
related tool


[14:39:59] SMILES Parse Error: syntax error while parsing: O=c1c(O)c(-c2c(C)cc(C(=O)N)cc2)oc2c(H)ccc(C(=O)N)c12
[14:39:59] SMILES Parse Error: check for mistakes around position 37:
[14:39:59] )cc(C(=O)N)cc2)oc2c(H)ccc(C(=O)N)c12
[14:39:59] ~~~~~~~~~~~~~~~~~~~~^
[14:39:59] SMILES Parse Error: Failed parsing SMILES 'O=c1c(O)c(-c2c(C)cc(C(=O)N)cc2)oc2c(H)ccc(C(=O)N)c12' for input: 'O=c1c(O)c(-c2c(C)cc(C(=O)N)cc2)oc2c(H)ccc(C(=O)N)c12'


=========== New best score: -8.9 for O=c1c(O)c(-c2c(C)cc(C(=O)N)cc2)oc2c(Cl)ccc(C(=O)N)c12 ========
Best score: -8.9 for O=c1c(O)c(-c2c(C)cc(C(=O)N)cc2)oc2c(Cl)ccc(C(=O)N)c12


In [11]:
start_chat()
scoring_args[1] = 'HMGCR'

date_string = time.strftime("%Y-%m-%d_%H-%M-%S")
filename = f'../results/ANT_FIRST/adversary_design_{date_string}.md'
with open(filename, 'w') as f:
    f.write(f'# Adversarial Design Session - test\n\n')

response_list = get_initial_prompt('HMGCR')
with open(filename, 'a') as f:
    f.write('# Initial model response:\n')
    text_av = response_list[-1][-2]
    f.write(text_av+'\n')

while text_av != 'Done':

    adv_response = adversary(text_av)
    with open(filename, 'a') as f:
        f.write('\n# Adversary feedback:\n')
        text_av = adv_response
        f.write(text_av +'\n')

    _, _, response_list = chat_turn(adv_response)
    with open(filename, 'a') as f:
        f.write('\n# Model response:\n')
        text_av = response_list[-1][-2]
        f.write(text_av +'\n')

lipinski tool
Starting grow cycle with best score -8.6 for O=c1cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12.
O=c1(I)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(Br)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(Cl)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(F)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C(Cl)(Cl)(Cl))cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C(F)(F)(F))cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C(=O)[O-])cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C(=O))cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C#N)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1([N+](=O)[O-])cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1([NH3+])cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(I)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(Br)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(Cl)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(F)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(C(Cl)(Cl)(Cl))ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(C(F)(F)(F))ccccc2)oc2cccc(C(C(=O)[O-

[10:27:32] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:27:32] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:27:32] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:27:32] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:27:32] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:27:32] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:27:32] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:27:32] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:27:32] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:27:32] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:27:32] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:27:32] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[10:27:32] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[10:27:32] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[10:27:32] 

O=c1c(I)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.2
O=c1c(Br)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.1
O=c1c(Cl)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.4
O=c1c(F)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -8.3
O=c1c(C(Cl)(Cl)(Cl))c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.2
O=c1c(C(F)(F)(F))c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.4
O=c1c(C(=O)[O-])c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.5
O=c1c(C(=O))c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.4
O=c1c(C#N)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.3
O=c1c([N+](=O)[O-])c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.0
O=c1c([NH3+])c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.9
O=c1cc(-c2c(I)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.5
O=c1cc(-c2c(Br)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.5
O=c1cc(-c2c(Cl)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2c(F)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2c(C(Cl)(Cl)(Cl))cccc2)oc2cccc(C(C(=O)[O-]))c12: -7.7
O=c1cc(-c2c(C(F)(F)(F))cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2c(C(=O)[O-])cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.

[10:47:29] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:47:29] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:47:29] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:47:29] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:47:29] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:47:29] Explicit valence for atom # 1 C, 6, is greater than permitted
[10:47:29] Can't kekulize mol.  Unkekulized atoms: 8 9 10 12 13
[10:47:29] Can't kekulize mol.  Unkekulized atoms: 9 10 11 13 14
[10:47:29] Can't kekulize mol.  Unkekulized atoms: 7 8 9 11 12
[10:47:29] Can't kekulize mol.  Unkekulized atoms: 8 9 10 12 13
[10:47:29] Can't kekulize mol.  Unkekulized atoms: 7 8 9 11 12
[10:47:29] Can't kekulize mol.  Unkekulized atoms: 9 10 11 13 14
[10:47:29] Can't kekulize mol.  Unkekulized atoms: 2 3 16 17 18 19 24
[10:47:29] Can't kekulize mol.  Unkekulized atoms: 2 3 17 18 19 20 25
[10:47:29] Can't kekulize mol.  Unkeku

O=c1c(C(=O)[O-])c(-c2ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.6
O=c1c(C(C(=O)[O-]))c(-c2ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.5
O=c1c(C#N)c(-c2ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.4
O=c1c([N+](=O)[O-])c(-c2ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.2
O=c1c(NC)c(-c2ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.1
O=c1c(C(F)(F)(F))c(-c2ccc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.3
O=c1cc(-c2c(C(=O)[O-])cc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.9
=========== New best score: -8.9 for O=c1cc(-c2c(C(=O)[O-])cc(F)cc2)oc2cccc(C(C(=O)[O-]))c12 ========
O=c1cc(-c2c(C(C(=O)[O-]))cc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.7
O=c1cc(-c2c(C#N)cc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2c([N+](=O)[O-])cc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.7
O=c1cc(-c2c(NC)cc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.1
O=c1cc(-c2c(C(F)(F)(F))cc(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.0
O=c1cc(-c2cc(C(=O)[O-])c(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2cc(C(C(=O)[O-]))c(F)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.4
O=c1cc(-c2cc(C#N)c(F)cc2)oc2cccc(C(C(=O

...

[11:33:33] Can't kekulize mol.  Unkekulized atoms: 2 3 16 17 21 22 27
[11:33:33] Can't kekulize mol.  Unkekulized atoms: 2 3 16 17 19 20 25
[11:33:33] Can't kekulize mol.  Unkekulized atoms: 2 3 16 17 19 20 25


related tool
got related molecules with smiles
(5,8-dioxonaphthalen-1-yl) 4-(1,2,2-trifluoroethenoxy)benzoate
lipinski tool
Starting grow cycle with best score -9.1 for O=c1cc(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12.
O=c1(C(=O)[O-])cc(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1([N+](=O)[O-])cc(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C#N)cc(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C(F)(F)(F))cc(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C(Cl)(Cl)(Cl))cc(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(C(=O)[O-])c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2([N+](=O)[O-])c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(C#N)c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(C(F)(F)(F))c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(C(Cl)(Cl)(Cl))c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2c(C(=O

[11:34:21] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:34:21] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:34:21] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:34:21] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:34:21] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:34:21] Can't kekulize mol.  Unkekulized atoms: 8 12 13 15 17
[11:34:21] Can't kekulize mol.  Unkekulized atoms: 8 12 13 15 17
[11:34:21] Can't kekulize mol.  Unkekulized atoms: 7 11 12 14 16
[11:34:21] Can't kekulize mol.  Unkekulized atoms: 9 13 14 16 18
[11:34:21] Can't kekulize mol.  Unkekulized atoms: 9 13 14 16 18
[11:34:21] Can't kekulize mol.  Unkekulized atoms: 2 3 20 21 22 23 28
[11:34:21] Can't kekulize mol.  Unkekulized atoms: 2 3 20 21 22 23 28
[11:34:21] Can't kekulize mol.  Unkekulized atoms: 2 3 19 20 21 22 27
[11:34:21] Can't kekulize mol.  Unkekulized atoms: 2 3 21 22 23 24 29
[11:34:21] Can't kekulize mol.

O=c1c(C(=O)[O-])c(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12: -7.6
O=c1c([N+](=O)[O-])c(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12: -7.3
O=c1c(C#N)c(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12: -7.4
O=c1c(C(F)(F)(F))c(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12: -7.9
O=c1c(C(Cl)(Cl)(Cl))c(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12: -7.8
O=c1cc(-c2c(C(=O)[O-])c(C(=O)[O-])c(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2c(C(=O)[O-])c([N+](=O)[O-])c(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12: -7.9
O=c1cc(-c2c(C(=O)[O-])c(C#N)c(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12: -8.1
O=c1cc(-c2c(C(=O)[O-])c(C(F)(F)(F))c(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12: -8.0
O=c1cc(-c2c(C(=O)[O-])c(C(Cl)(Cl)(Cl))c(F)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12: -7.8
O=c1cc(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2c(C(=O)[O-])ccc(C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2c([N+](=O)[O-])ccc(C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2c(C(=O)[O-])cc(F)c(Cl)c2)oc2c(C#N)ccc(C(C(=O)[O-]))c12: -8.

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.1: `tool_use` ids were found without `tool_result` blocks immediately after: toolu_01JDuPGgyRM5Eh85g3yut28b. Each `tool_use` block must have a corresponding `tool_result` block in the next message.'}, 'request_id': 'req_011CZczV1g2rNmkkZKv9Tux9'}

## Lipinski test

In [ ]:
openai_key = os.getenv("OPENAI_API_TOKEN")

tools = [grow_cycle, replace_groups, make_random_list]

model = ChatOpenAI(model_name="gpt-5.2", api_key=openai_key).bind_tools(tools)

class State(TypedDict):
  messages: Annotated[list, add_messages]

def model_node(state: State) -> State:
  res = model.invoke(state['messages'])
  return {'messages': res}

builder = StateGraph(State)
builder.add_node('model', model_node)
builder.add_node('tools', ToolNode(tools))
builder.add_edge(START, 'model')
builder.add_conditional_edges('model', tools_condition)
builder.add_edge('tools',  'model')

graph = builder.compile()

sys_message = SystemMessage(content=f'''
{task_specific_prompt}

## You will first:
- Read the list of molecule SMILES and scores
- Ascertain any features of the molecules that contribute to the desired score. For example, if,
from one molecule to the next, the addition of an O group makes the score better.
- Gather all of these trends across all of the molecules.

## If you need additional information to ascertain the trends, such as more modified
molecules and their docking scores, you have tools you can call to generate new
molecules and get their docking scores. You can use these tools as many times as you want
to gather information on the trends.

the tools include:
                            
- grow_cycle: starts with a molecule SMILES and adds substituents to it, docks them, and returns 
              a list of molecules and scores. You can use this tool to further explore modifications
              to promising molecules that you find in the input data. You can provide a list of
              substituents to add, or use the predefined sets: e_withdraw (electron withdrawing),
              e_donate (electron donating), withdraw_with_linkers (electron withdrawing with linkers), 
              donate_with_linkers (electron donating with linkers). You can also generate a random list 
              of substituents with the make_random_list tool and use that as input to grow_cycle.

- replace_groups: starts with a molecule SMILES and replaces specific groups in it with new groups, returning a list of new
                  molecules and scores. This tool allows you to test specific hypotheses about how replacing certain
                  groups in a molecule might affect binding affinity. You can specify which groups to replace and
                  what to replace them with, or use the predefined sets of substituents mentioned above. You can also 
                  generate a random list of substituents with the make_random_list tool and use that as input to replace_groups.

- make_random_list: this tool generates a list of substituents of specified length (num_items). It draws from the predefined lists:
                    e_withdraw (electron withdrawing), e_donate (electron donating), withdraw_with_linkers 
                    (electron withdrawing with linkers), donate_with_linkers (electron donating with linkers). 
                    Use this tool when you want to get a broad sense of how different modifications affect binding affinity, 
                    without having a specific hypothesis in mind.

## Once you have ascertained the trends:
- Use the trends you learned to suggest 1-5 new molecules that obey the trends you found
and which should have a better score than the molecules in the list.
- Provide reasoning as to why you created those new molecules.
- Estimate the new scores.

## You may ask the user for clarification if needed, but try to use the tools to gather as much information as you
can before asking for clarification.

## In further turns, you will also receive feedback from an adversary model that is trying to find flaws 
in your reasoning and suggest improvements to your proposed molecules. You should use this feedback to 
refine your understanding of the trends, run new experiments with the tools to gather more information, 
and improve your proposed molecules in subsequent turns.

## Once you have reached a point where you think you have proposed the best possible molecules based on 
the trends, tool results and the adversary feedback, reply with only one word: "Done". This will signal that you have 
finished the task and will not propose any more molecules.
''')

global messages
messages = [sys_message]

#@spaces.GPU
def start_chat():
  '''
  '''
  global chat_history, messages, reasoning
  chat_history = []
  reasoning = []
  messages = [sys_message]

#@spaces.GPU
def chat_turn(prompt: str):
  '''
  '''
  global chat_history, messages, reasoning
  human_message = HumanMessage(content=prompt)
  messages.append(human_message)
  local_history = [prompt]

  input = {
      'messages' : messages
  }

  for c in graph.stream(input):
    try:
      ai_mes = c['model']['messages'].content
      messages.append(AIMessage(ai_mes))
      if ai_mes != '':
        print(f'message is {ai_mes}')
        local_history.append(ai_mes)
    except:
      pass
    try:
      if os.path.exists('current_image.png'):
        if os.path.getmtime('current_image.png') > time.time() - 30:
          img = Image.open('current_image.png')
        else:
          img = None
      else:
        img = None
    except:
      img = None
    try:
      reasoning.append(c['tools']['messages'][0].content)
    except:
      pass

  if len(local_history) != 2:
    local_history.append('no message')

  #chat_history.append({'role': 'user', 'content': local_history[0]})
  #chat_history.append({'role': 'assistant', 'content': local_history[1]})
  chat_history.append(local_history)
  return '', img, chat_history

def get_initial_prompt(protein):
  '''
  '''
  scoring_args[1] = protein
  random_list, excluded_list = make_random_list(10)
  first_list = sub_cycle(random_list, scoring_args)
  context = ''
  for smiles, score in first_list:
      context += f"{smiles}: {score}\n"

  first_prompt = f'''
  Here is a list of molecules and their scores:
  {context}\n'''

  blank, img, mes = chat_turn(first_prompt)
  #print(mes[-1])
  
  return mes

dudes = ['IGF1R', 'JAK2', 'KIT', 'LCK', 'MAPK14', 'MAPKAPK2', 'MET', 'PTK2', 'PTPN1', 'SRC', 'ABL1', 'AKT1', 'AKT2', 'CDK2', 'CSF1R', 'EGFR', 'KDR', 'MAPK1', 'FGFR1', 'ROCK1', 'MAP2K1', 'PLK1',
         'HSD11B1', 'PARP1', 'PDE5A', 'PTGS2', 'ACHE', 'MAOB', 'CA2', 'GBA', 'HMGCR', 'NOS1', 'REN', 'DHFR', 'ESR1', 'ESR2', 'NR3C1', 'PGR', 'PPARA', 'PPARD', 'PPARG',
        'AR','THRB','ADAM17', 'F10', 'F2', 'BACE1', 'CASP3', 'MMP13', 'DPP4', 'ADRB1', 'ADRB2', 'DRD2', 'DRD3','ADORA2A','CYP2C9', 'CYP3A4', 'HSP90AA1']


In [19]:
import importlib
importlib.reload(sys.modules['lipinski_module'])
from lipinski_module import *
importlib.reload(sys.modules['MolPropOp'])
from MolPropOp import *

In [24]:
start_chat()

In [25]:
get_initial_prompt('alogp')

Starting sub cycle on base rings for protein alogp.
c1(CC=C(I))ccccc1: 3.1778000000000013
c1(N(F))ccccc1: 1.9829999999999999
c1(C(=O)O(C#N))ccccc1: 1.3244799999999999
c1(CC([Si](C)(C)C))ccccc1: 3.567300000000002
c1(Br)ccccc1: 2.4491000000000005
c1(S(c5ccccc5))ccccc1: 3.8378000000000014
c1(CC=C(O))ccccc1: 2.3008000000000006
c1(C=CC(C(Cl)(Cl)(Cl)))ccccc1: 4.460100000000002
c1(C=C(C(C)C))ccccc1: 3.355800000000002
c1(C(=O)(OC(=O)C))ccccc1: 1.3899000000000001
n1c(CC=C(I))cccc1: 2.572800000000001
n1c(N(F))cccc1: 1.378
n1c(C(=O)O(C#N))cccc1: 0.7194799999999999
n1c(CC([Si](C)(C)C))cccc1: 2.9623000000000017
n1c(Br)cccc1: 1.8440999999999999
n1c(S(c5ccccc5))cccc1: 3.232800000000001
n1c(CC=C(O))cccc1: 1.6958
n1c(C=CC(C(Cl)(Cl)(Cl)))cccc1: 3.855100000000002
n1c(C=C(C(C)C))cccc1: 2.750800000000001
n1c(C(=O)(OC(=O)C))cccc1: 0.7848999999999999
n1cc(CC=C(I))ccc1: 2.572800000000001
n1cc(N(F))ccc1: 1.378
n1cc(C(=O)O(C#N))ccc1: 0.7194799999999999
n1cc(CC([Si](C)(C)C))ccc1: 2.9623000000000017
n1cc(Br)ccc1:

[['\n  Here is a list of molecules and their scores:\n  c1(CC=C(I))ccccc1: 3.1778000000000013\nc1(N(F))ccccc1: 1.9829999999999999\nc1(C(=O)O(C#N))ccccc1: 1.3244799999999999\nc1(CC([Si](C)(C)C))ccccc1: 3.567300000000002\nc1(Br)ccccc1: 2.4491000000000005\nc1(S(c5ccccc5))ccccc1: 3.8378000000000014\nc1(CC=C(O))ccccc1: 2.3008000000000006\nc1(C=CC(C(Cl)(Cl)(Cl)))ccccc1: 4.460100000000002\nc1(C=C(C(C)C))ccccc1: 3.355800000000002\nc1(C(=O)(OC(=O)C))ccccc1: 1.3899000000000001\nn1c(CC=C(I))cccc1: 2.572800000000001\nn1c(N(F))cccc1: 1.378\nn1c(C(=O)O(C#N))cccc1: 0.7194799999999999\nn1c(CC([Si](C)(C)C))cccc1: 2.9623000000000017\nn1c(Br)cccc1: 1.8440999999999999\nn1c(S(c5ccccc5))cccc1: 3.232800000000001\nn1c(CC=C(O))cccc1: 1.6958\nn1c(C=CC(C(Cl)(Cl)(Cl)))cccc1: 3.855100000000002\nn1c(C=C(C(C)C))cccc1: 2.750800000000001\nn1c(C(=O)(OC(=O)C))cccc1: 0.7848999999999999\nn1cc(CC=C(I))ccc1: 2.572800000000001\nn1cc(N(F))ccc1: 1.378\nn1cc(C(=O)O(C#N))ccc1: 0.7194799999999999\nn1cc(CC([Si](C)(C)C))ccc1: 2.962

## HOMO_LUMO

In [1]:
import sys
sys.path.append('code')
from HL_gap_module import *

Number of cores: 2


In [3]:
scoring_function('CCuO')

Could not calculate gap


[12:37:32] SMILES Parse Error: syntax error while parsing: CCuO
[12:37:32] SMILES Parse Error: check for mistakes around position 3:
[12:37:32] CCuO
[12:37:32] ~~^
[12:37:32] SMILES Parse Error: Failed parsing SMILES 'CCuO' for input: 'CCuO'


(100.0, None)